# 01 - Data Understanding
Segmentasi Pelanggan E-Commerce (RFM + K-Means)

Notebook ini menjalankan pemeriksaan kualitas data (bagian 2.4) dan 8 analisis EDA (bagian 2.5) pada laporan. Seluruh grafik pada bagian EDA dihasilkan dari notebook ini.

In [3]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rfm_kmeans import config
cfg = config.load_config()
SEED = cfg["seed"]
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:,.2f}".format)
from rfm_kmeans import data_loader, sampling, visualization
FIG = config.resolve(cfg, "figures_dir")

## 1. Muat data mentah

In [4]:
raw = data_loader.load_raw(config.resolve(cfg, "raw_csv"), config.resolve(cfg, "raw_xlsx"))
print(raw.shape)
raw.head()

(541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [5]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [6]:
raw.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,"541,909.00",NaN,NaN,NaN,9.55,"-80,995.00",1.00,3.00,10.00,"80,995.00",218.08
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,"541,909.00",NaN,NaN,NaN,4.61,"-11,062.06",1.25,2.08,4.13,"38,970.00",96.76
CustomerID,"406,829.00",NaN,NaN,NaN,"15,287.69","12,346.00","13,953.00","15,152.00","16,791.00","18,287.00","1,713.60"
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Pemeriksaan kualitas data (2.4)

In [7]:
# Missing value
missing = raw.isna().sum().to_frame("jumlah")
missing["persen"] = 100 * missing["jumlah"] / len(raw)
missing

,jumlah,persen
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00


In [8]:
# Duplikasi
print("Baris duplikat:", raw.duplicated().sum())

# Pembatalan
is_cancel = raw["InvoiceNo"].astype(str).str.upper().str.startswith("C")
print("Baris pembatalan (C):", is_cancel.sum(), f"({100*is_cancel.mean():.2f}%)")

# Nilai tidak valid
print("Quantity <= 0:", (raw["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (raw["UnitPrice"] <= 0).sum())

Baris duplikat: 5268
Baris pembatalan (C): 9288 (1.71%)
Quantity <= 0: 10624
UnitPrice <= 0: 2517


In [9]:
# Kode non-produk (diperiksa manual sebelum dikeluarkan)
suspect = raw[~raw["StockCode"].astype(str).str.fullmatch(r"\d{5}[A-Za-z]*")]
suspect.groupby(["StockCode", "Description"]).size().sort_values(ascending=False).head(20)

StockCode     Description                       
POST          POSTAGE                               1252
DOT           DOTCOM POSTAGE                         709
M             Manual                                 571
C2            CARRIAGE                               143
D             Discount                                77
S             SAMPLES                                 63
BANK CHARGES  Bank Charges                            37
AMAZONFEE     AMAZON FEE                              34
CRUK          CRUK Commission                         16
DCGSSGIRL     GIRLS PARTY BAG                         13
DCGSSBOY      BOYS PARTY BAG                          11
gift_0001_20  Dotcomgiftshop Gift Voucher £20.00       9
gift_0001_10  Dotcomgiftshop Gift Voucher £10.00       8
gift_0001_30  Dotcomgiftshop Gift Voucher £30.00       7
gift_0001_50  Dotcomgiftshop Gift Voucher £50.00       4
DCGS0003      BOXED GLASS ASHTRAY                      4
PADS          PADS TO MATCH ALL CUSHION

In [10]:
# Outlier (aturan IQR)
for col in ["Quantity", "UnitPrice"]:
    q1, q3 = raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((raw[col] < q1 - 1.5*iqr) | (raw[col] > q3 + 1.5*iqr)).sum()
    print(f"{col}: {n_out:,} outlier ({100*n_out/len(raw):.2f}%)")

Quantity: 58,619 outlier (10.82%)
UnitPrice: 39,627 outlier (7.31%)


## 3. Sampling per CustomerID dan representativitas (2.6)

In [11]:
sample = sampling.sample_by_customer(raw, cfg["sampling"]["target_rows"], SEED)
print(f"Sampel: {len(sample):,} baris dari {len(raw):,} | pelanggan: {sample['CustomerID'].nunique():,}")
sampling.compare_distribution(raw, sample, "Country")

Sampel: 50,230 baris dari 541,909 | pelanggan: 596


,penuh,sampel
Country,,
United Kingdom,0.91,0.92
Germany,0.02,0.01
France,0.02,0.03
EIRE,0.02,0.00
Spain,0.00,0.00


Bandingkan juga distribusi `Quantity` dan `UnitPrice` antara data penuh dan sampel. Ulangi sampling bila menyimpang jauh.

In [12]:
pd.DataFrame({"penuh": raw[["Quantity", "UnitPrice"]].median(),
              "sampel": sample[["Quantity", "UnitPrice"]].median()})

,penuh,sampel
Quantity,3.00,6.00
UnitPrice,2.08,1.79


## 4. EDA (2.5)
Analisis dilakukan pada data yang sudah dibersihkan secara ringan agar grafik tidak didominasi baris tidak valid.

In [13]:
from rfm_kmeans import cleaning, features
clean, log = cleaning.clean_transactions(sample, cfg["cleaning"])
log

{'awal': 50230,
 'setelah_hapus_CustomerID_kosong': 50230,
 'setelah_hapus_duplikat': 49548,
 'setelah_hapus_pembatalan': 48293,
 'setelah_hapus_nilai_tidak_valid': 48290,
 'setelah_hapus_non_produk': 48150,
 'akhir': 48150,
 'persen_tersisa': 95.86}

### EDA 1 - Jumlah transaksi per bulan

In [14]:
visualization.plot_monthly_transactions(clean, FIG / "eda1_monthly.png")
plt.show()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/1141407596.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### EDA 2 - Distribusi Quantity dan UnitPrice

In [15]:
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for i, col in enumerate(["Quantity", "UnitPrice"]):
    sns.histplot(clean[col], bins=60, ax=ax[0, i]); ax[0, i].set_yscale("log"); ax[0, i].set_title(f"Histogram {col}")
    sns.boxplot(x=clean[col], ax=ax[1, i]); ax[1, i].set_title(f"Boxplot {col}")
plt.tight_layout(); fig.savefig(FIG / "eda2_quantity_price.png", dpi=150); plt.show()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/2902922897.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); fig.savefig(FIG / "eda2_quantity_price.png", dpi=150); plt.show()


### EDA 3 - Jumlah transaksi per negara

In [16]:
top = clean.groupby("Country")["InvoiceNo"].nunique().sort_values(ascending=False).head(10)
ax = top.plot(kind="barh", figsize=(8, 4), title="Transaksi per negara (top 10)")
ax.invert_yaxis(); plt.tight_layout(); plt.savefig(FIG / "eda3_country.png", dpi=150); plt.show()
print(f"Porsi negara teratas: {100*top.iloc[0]/clean['InvoiceNo'].nunique():.1f}%")

Porsi negara teratas: 93.1%


/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/3134935980.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.invert_yaxis(); plt.tight_layout(); plt.savefig(FIG / "eda3_country.png", dpi=150); plt.show()


### EDA 4 - Transaksi dan total belanja per pelanggan (skala log)

In [17]:
clean = clean.assign(TotalPrice=clean["Quantity"] * clean["UnitPrice"])
per_cust = clean.groupby("CustomerID").agg(n_invoice=("InvoiceNo", "nunique"), total=("TotalPrice", "sum"))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(per_cust["n_invoice"], bins=50, log_scale=True, ax=ax[0]); ax[0].set_title("Jumlah invoice per pelanggan")
sns.histplot(per_cust["total"], bins=50, log_scale=True, ax=ax[1]); ax[1].set_title("Total belanja per pelanggan")
plt.tight_layout(); fig.savefig(FIG / "eda4_per_customer.png", dpi=150); plt.show()

# Konsentrasi belanja: berapa % pendapatan dari 20% pelanggan teratas?
s = per_cust["total"].sort_values(ascending=False)
print(f"20% pelanggan teratas menyumbang {100*s.head(int(0.2*len(s))).sum()/s.sum():.1f}% pendapatan")

20% pelanggan teratas menyumbang 71.3% pendapatan


/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/3705780255.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); fig.savefig(FIG / "eda4_per_customer.png", dpi=150); plt.show()


### EDA 5 - Distribusi Recency, Frequency, Monetary

In [18]:
rfm = features.build_rfm(clean, cfg["rfm"]["reference_offset_days"])
visualization.plot_rfm_distribution(rfm, FIG / "eda5_rfm_distribution.png")
plt.show()
rfm[["Recency", "Frequency", "Monetary"]].agg(["mean", "median", "std", "skew"]).T

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/1794572911.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,mean,median,std,skew
Recency,89.67,46.00,100.63,1.38
Frequency,4.18,2.00,5.72,5.13
Monetary,"1,841.35",713.89,"5,981.73",11.09


Skewness positif yang besar pada Frequency dan Monetary menandakan perlunya transformasi log sebelum K-Means.

### EDA 6 - Korelasi antar fitur RFM

In [19]:
visualization.plot_rfm_correlation(rfm, FIG / "eda6_rfm_correlation.png")
plt.show()
np.log1p(rfm[["Recency", "Frequency", "Monetary"]]).corr()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/3158081882.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Recency,Frequency,Monetary
Recency,1.00,-0.55,-0.44
Frequency,-0.55,1.00,0.80
Monetary,-0.44,0.80,1.00


### EDA 7 - Produk terlaris

In [20]:
best = clean.groupby("Description")["Quantity"].sum().sort_values(ascending=False).head(10)
ax = best.plot(kind="barh", figsize=(9, 4), title="10 produk terlaris (unit terjual)")
ax.invert_yaxis(); plt.tight_layout(); plt.savefig(FIG / "eda7_top_products.png", dpi=150); plt.show()

/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/936946052.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.invert_yaxis(); plt.tight_layout(); plt.savefig(FIG / "eda7_top_products.png", dpi=150); plt.show()


### EDA 8 - Proporsi transaksi pembatalan
Dihitung dari sampel mentah (sebelum pembatalan dibuang).

In [21]:
s = sample.copy()
s["Status"] = np.where(s["InvoiceNo"].astype(str).str.upper().str.startswith("C"), "Pembatalan", "Normal")
share = s.groupby("Status")["InvoiceNo"].nunique()
ax = share.plot(kind="bar", rot=0, title="Invoice normal vs pembatalan")
plt.tight_layout(); plt.savefig(FIG / "eda8_cancellations.png", dpi=150); plt.show()
print((100 * share / share.sum()).round(2))

Status
Normal       83.01
Pembatalan   16.99
Name: InvoiceNo, dtype: float64


/var/folders/82/c0ftlfcj0tv7njdzrpnwnbxh0000gn/T/ipykernel_95659/1190553362.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(FIG / "eda8_cancellations.png", dpi=150); plt.show()


## 5. Ringkasan temuan
Isi setelah notebook dijalankan:
- Persentase CustomerID kosong: ...
- Persentase pembatalan: ...
- Skewness R / F / M: ...
- Korelasi Frequency - Monetary: ...
- Keputusan: transformasi log + StandardScaler diperlukan / tidak diperlukan karena ...